# NVIDIA OpenShell + Deep Agents + Nemotron Ultra Incident Analysis Workshop

You will build and run a Deep Agent powered by the recently announced NVIDIA Nemotron 3 Ultra 550B-A55B model through NVIDIA NIM. The agent analyzes application logs by writing Python code inside an NVIDIA OpenShell sandbox, then you inspect the full execution in LangSmith.

The sample data in `workshop/data/sample_app.log` is a synthetic application log from a small ecommerce-style system with frontend, auth, inventory, checkout, and payments services. It contains normal requests, latency warnings, repeated inventory database timeouts, checkout failures, status codes, request IDs, timestamps, service names, paths, and latency values.

The agent should parse those logs, compute evidence about the incident, identify the likely failure pattern, write an incident summary JSON file, write a Markdown report, and explain what happened.

## What You Will Learn

- **NVIDIA Nemotron 3 Ultra via NIM**: the default model that drives the Deep Agent.
- **Deep Agents**: planning, subagents, skills, and file tools around the model.
- **NVIDIA OpenShell**: executing agent-written Python inside a governed sandbox instead of on your host machine.
- **Generic sandbox tooling**: using Deep Agents' built-in file and execution tools, not only prepared workshop tools.
- **LangSmith**: inspecting traces for model calls, subagent handoffs, tool calls, command output, errors, and metadata.
- **Policy observability**: seeing one sandbox write allowed and one protected filesystem write denied.

The sandbox matters here because the agent is not just summarizing text. It is writing code, running that code, creating artifacts, and operating inside security boundaries.

## 0. Load Local Configuration

This cell loads `.env` into the notebook process without printing secret values. It also confirms you are running from the repo root.

In [ ]:
import os
import subprocess
import time
import traceback
import uuid
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "workshop/notebooks/participant_walkthrough.ipynb").exists():
            return candidate.resolve()
    raise RuntimeError("Could not find the repo root. Open this notebook from inside the DeepAgents_OpenShell checkout.")

ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)
print("Repo root:", ROOT)

def load_env_file(path: Path = ROOT / ".env") -> list[str]:
    if not path.exists():
        raise FileNotFoundError(".env is missing. Copy .env.example to .env and fill in local values.")
    loaded = []
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        name = name.strip()
        value = value.strip().strip('"').strip("'")
        if name and value and name not in os.environ:
            os.environ[name] = value
            loaded.append(name)
    return loaded

loaded_names = load_env_file()
os.environ.setdefault("LANGCHAIN_CALLBACKS_BACKGROUND", "false")
os.environ.setdefault("GRPC_VERBOSITY", "ERROR")
print(f"Loaded {len(loaded_names)} names from .env. Values are hidden.")
print("LangSmith project:", os.environ.get("LANGSMITH_PROJECT", "<missing>"))
print("LangChain callbacks background:", os.environ.get("LANGCHAIN_CALLBACKS_BACKGROUND", "<unset>"))

## 1. Preflight

The preflight checks Python, Docker, required packages, OpenShell gateway reachability, and required config names. It never prints secret values.

Important: Docker running is necessary but not sufficient. OpenShell also needs a configured gateway. If this cell reports `OpenShell gateway reachable` as missing, stop here and configure/register an OpenShell gateway before continuing. Without that gateway, the agent cannot create or use a real sandbox.

In [ ]:
def run_command(command: list[str], *, check: bool = False, timeout: int | None = None) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(command))
    result = subprocess.run(command, text=True, capture_output=True, timeout=timeout)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(command)}")
    return result

run_command(["uv", "run", "python", "scripts/check_env.py"], check=True)

## 2. Inspect The Sample Incident Log

This is the file the agent will analyze. It contains a short checkout incident: inventory timeouts cascade into checkout failures, then the system recovers.

A plain LLM could summarize a tiny pasted snippet. The point of this workshop is different: have the agent operate over a file, write Python, compute evidence, and create artifacts in a sandbox.

In [ ]:
log_path = ROOT / "workshop/data/sample_app.log"
lines = log_path.read_text(encoding="utf-8").splitlines()
print(f"Sample log lines: {len(lines)}")
print("\n".join(lines[:8]))

## 3. Prepare The OpenShell Sandbox

The workshop uses a policy fixture that allows writes under `/sandbox` and keeps protected system paths such as `/etc` read-only. This gives us a deterministic policy-observability lab later.

If the sandbox already exists, the create command may fail. That is okay; the cell will then apply the policy to the existing sandbox. The final readiness check runs a tiny command inside the sandbox, so this cell only prints `Sandbox ready` after OpenShell can actually execute code there.


In [ ]:
sandbox_name = os.environ.get("OPENSHELL_SANDBOX_NAME", "workshop-sandbox")
policy_path = "workshop/policies/local-sandbox.yaml"

status = run_command(["uv", "run", "openshell", "status"], check=True)
status_text = f"{status.stdout}\n{status.stderr}"
gateway_blockers = (
    "No gateway configured",
    "Set one with: openshell gateway select",
    "Register a gateway with:",
    "failed to connect",
    "connection refused",
)
if any(marker in status_text for marker in gateway_blockers):
    raise RuntimeError(
        "OpenShell CLI is installed, but no reachable gateway is configured. "
        "Register/select a gateway, then rerun the preflight before creating the sandbox."
    )

try:
    create = run_command([
        "uv", "run", "openshell", "sandbox", "create",
        "--name", sandbox_name,
        "--policy", policy_path,
        "--no-tty",
        "--",
        "true",
    ], check=False, timeout=180)
except subprocess.TimeoutExpired as exc:
    raise RuntimeError(
        "Timed out waiting for OpenShell sandbox creation. "
        "If the active gateway is a cloud VM and the output stopped at 'Waiting for supervisor relay', "
        "select your local gateway for the main workshop labs or inspect the VM gateway logs before continuing."
    ) from exc

create_text = f"{create.stdout}\n{create.stderr}"
if any(marker in create_text for marker in gateway_blockers + ("No active gateway",)):
    raise RuntimeError(
        "OpenShell could not create a sandbox because no active gateway is configured. "
        "Fix the gateway first, then rerun this cell."
    )

if create.returncode != 0:
    print("Sandbox create did not complete cleanly. Applying policy to the configured sandbox instead.")
    run_command([
        "uv", "run", "openshell", "policy", "set",
        sandbox_name,
        "--policy", policy_path,
        "--wait",
    ], check=True, timeout=120)

ready = run_command([
    "uv", "run", "openshell", "sandbox", "exec",
    "--name", sandbox_name,
    "--no-tty",
    "--",
    "python3", "--version",
], check=False, timeout=60)
if ready.returncode != 0:
    raise RuntimeError(
        "OpenShell sandbox exists but is not executable yet. "
        "Do not continue until `openshell sandbox exec --name " + sandbox_name + " --no-tty -- python3 --version` succeeds. "
        "If the active gateway is the cloud VM path, switch back to the local gateway for the main labs or debug the VM gateway readiness issue."
    )

print("Sandbox ready:", sandbox_name)


## 4. Import The Deep Agent

The agent is created with `create_deep_agent(...)`. The important idea is that the model is only one part of the system: Deep Agents adds tools, skills, subagents, and a filesystem/runtime backend around the model.

In this workshop configuration:

- The **orchestrator** reads the top-level system prompt and decides which tool or subagent should handle the request.
- `sandbox-engineer` handles OpenShell command execution, Python scripts, and file roundtrips.
- `policy-analyst` handles allowed/denied operation analysis.
- `observability-coach` handles LangSmith debugging guidance.

For the full agent breakdown and anatomy, open `src/openshell_obs_workshop/agent.py`. That file shows the `create_deep_agent(...)` call, top-level tools, subagents, skills, model, and OpenShell backend wiring.

The next cell loads the real agent object, prints the configuration summary, and shows the top-level system prompt so you can connect the notebook behavior to the Deep Agents setup code.

### Runtime Boundary

A useful mental model for the rest of the workbook:

- The notebook, model client, and Deep Agent orchestrator run on your laptop.
- Deep Agents contributes built-in tools such as `write_todos`, `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, and `execute`.
- The OpenShell backend makes `execute` and sandbox file operations happen inside the selected OpenShell sandbox.
- `/sandbox` is sandbox storage. `/skills/` and `/workshop/` are local virtual routes backed by this repo so participants can inspect and edit workshop material.



In [ ]:
from openshell_obs_workshop.agent import _agent_kwargs, agent


def tool_name(tool) -> str:
    return getattr(tool, "name", getattr(tool, "__name__", type(tool).__name__))

def render_markdown(text: str) -> None:
    get_ipython_fn = globals().get("get_ipython")
    shell = get_ipython_fn() if get_ipython_fn else None
    if shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell":
        from IPython.display import Markdown, display
        display(Markdown(text))
    else:
        print(text)

print("Loaded agent:", type(agent).__name__)
print("Agent name:", _agent_kwargs["name"])
print("Model class:", type(_agent_kwargs["model"]).__name__)
print("Model provider:", os.environ.get("MODEL_PROVIDER", "nvidia"))
print("Configured NVIDIA model:", os.environ.get("NVIDIA_MODEL", "nvidia/nemotron-3-ultra-550b-a55b"))
print("Backend:", f"{type(_agent_kwargs['backend']).__name__} (OpenShell sandbox plus local workshop/skills routes)")
print("Workshop tools:", ", ".join(tool_name(tool) for tool in _agent_kwargs["tools"]))
print("Subagents:", ", ".join(subagent["name"] for subagent in _agent_kwargs["subagents"]))
print("Skills:", ", ".join(_agent_kwargs["skills"]))

render_markdown("### Top-level system prompt\n\n```text\n" + _agent_kwargs["system_prompt"] + "\n```")

subagent_lines = [
    f"- `{subagent['name']}`: {subagent['description']}"
    for subagent in _agent_kwargs["subagents"]
]
render_markdown("### Subagent routing\n\n" + "\n".join(subagent_lines))

render_markdown(
    "### Built-in Deep Agents tools\n\n"
    "The `Workshop tools` line above lists only the custom tools this repo adds. "
    "Deep Agents also provides built-in file, shell, todo, and subagent tools: "
    "`write_todos`, `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, `execute`, and `task`. "
    "Those built-ins use the configured backend, so shell execution and sandbox file operations go through OpenShell."
)


## 5. Invocation Helper

This helper invokes the agent and prints only the final response. LangSmith will receive the detailed trace if `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` is configured.

In [ ]:
def final_text(result) -> str:
    message = result["messages"][-1]
    content = getattr(message, "content", None)
    if content is None and isinstance(message, dict):
        content = message.get("content", "")
    if isinstance(content, list):
        return "\n".join(str(part) for part in content)
    return str(content)

def is_transient_agent_error(exc: Exception) -> bool:
    text = "".join(traceback.format_exception(type(exc), exc, exc.__traceback__))
    markers = (
        "RemoteDisconnected",
        "Remote end closed connection",
        "Connection aborted",
        "ConnectionError",
        "ReadTimeout",
        "ServiceUnavailable",
        "Too Many Requests",
        "rate limit",
        "temporarily unavailable",
    )
    return any(marker.lower() in text.lower() for marker in markers)

def tracing_configured() -> bool:
    flag = os.environ.get("LANGSMITH_TRACING", "").strip().lower()
    return flag in {"1", "true", "yes", "on"} and bool(os.environ.get("LANGSMITH_API_KEY"))

def langsmith_trace_url(run_id: uuid.UUID, *, attempts: int = 8, delay_seconds: float = 0.75) -> str | None:
    if not tracing_configured():
        return None
    try:
        import logging
        from contextlib import redirect_stderr, redirect_stdout
        from io import StringIO
    except Exception as exc:
        print(f"LangSmith trace URL unavailable: {type(exc).__name__}: {exc}")
        return None

    sdk_output = StringIO()
    previous_logging_disable_level = logging.root.manager.disable
    logging.disable(logging.CRITICAL)
    try:
        with redirect_stdout(sdk_output), redirect_stderr(sdk_output):
            from langsmith import Client
            client = Client()
    except Exception as exc:
        print(f"LangSmith trace URL unavailable. Last error type: {type(exc).__name__}")
        return None
    finally:
        logging.disable(previous_logging_disable_level)
    project_name = os.environ.get("LANGSMITH_PROJECT") or None
    for attempt in range(1, attempts + 1):
        try:
            previous_logging_disable_level = logging.root.manager.disable
            logging.disable(logging.CRITICAL)
            with redirect_stdout(sdk_output), redirect_stderr(sdk_output):
                client.flush(timeout=5)
                run = client.read_run(run_id)
                return client.get_run_url(run=run, project_name=project_name)
        except Exception as exc:
            if attempt >= attempts:
                print(f"LangSmith trace URL not ready yet for run id {run_id}. Last error type: {type(exc).__name__}")
                return None
            time.sleep(delay_seconds)
        finally:
            logging.disable(previous_logging_disable_level)
    return None

def print_langsmith_trace_link(run_id: uuid.UUID) -> None:
    if not tracing_configured():
        return
    url = langsmith_trace_url(run_id)
    if not url:
        print("LangSmith trace link unavailable. Run id:", run_id)
        return
    get_ipython_fn = globals().get("get_ipython")
    shell = get_ipython_fn() if get_ipython_fn else None
    if shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell":
        from IPython.display import Markdown, display
        display(Markdown(f"**LangSmith trace:** [open this run]({url})"))
    else:
        print("LangSmith trace:", url)

agent_outputs: dict[str, str] = {}

def run_agent(prompt: str, *, lab: str, attempts: int = 3) -> None:
    config = {
        "configurable": {},
        "tags": ["workshop", lab, "notebook", "incident-analysis"],
        "metadata": {
            "workshop": "openshell-deepagents-langsmith",
            "lab": lab,
            "sandbox_name": os.environ.get("OPENSHELL_SANDBOX_NAME", "workshop-sandbox"),
            "use_case": "incident-log-analysis",
        },
    }
    last_error: Exception | None = None
    for attempt in range(1, attempts + 1):
        run_id = uuid.uuid4()
        thread_id = f"notebook-{lab}-{uuid.uuid4().hex[:8]}-a{attempt}"
        attempt_config = {
            **config,
            "configurable": {"thread_id": thread_id},
            "run_id": run_id,
            "run_name": f"workshop-{lab}",
        }
        try:
            result = agent.invoke({"messages": [{"role": "user", "content": prompt}]}, config=attempt_config)
            text = final_text(result)
            agent_outputs[lab] = text
            print(text)
            print()
            print_langsmith_trace_link(run_id)
            return None
        except Exception as exc:
            last_error = exc
            if attempt >= attempts or not is_transient_agent_error(exc):
                raise
            print(f"Transient agent/provider error on attempt {attempt}; retrying with a new thread id...")
    raise RuntimeError("Agent invocation failed") from last_error

## Lab 1: Sandbox Smoke Test

This run checks that the agent can execute commands in OpenShell.

After the cell runs, open LangSmith and inspect:

- root run
- subagent handoff
- OpenShell command span
- final answer

In [ ]:
run_agent(
    "Use the run_sandbox_smoke_test tool. Summarize the OS, architecture, "
    "and Python version from the OpenShell sandbox output.",
    lab="sandbox-smoke-test",
)

## Lab 2: Python Incident Analysis

This is the core workshop use case. The agent uses a purpose-built incident-analysis tool that writes a Python parser into OpenShell, executes it in the sandbox, and produces two artifacts:

- `/sandbox/incident_summary.json`
- `/sandbox/incident_report.md`

The important learning point is that the model is orchestrating a tool, not doing all of the analysis inside a chat response. In this lab:

1. The Deep Agent decides to call `analyze_incident_log`.
2. The tool reads `workshop/data/sample_app.log` from the repo.
3. The tool copies that log into OpenShell at `/sandbox/sample_app.log`.
4. The tool writes a Python script into OpenShell at `/sandbox/analyze_incident.py`.
5. OpenShell runs that script inside the sandbox, not on your host machine.
6. The script parses records, counts errors and warnings, computes service latency, identifies failing endpoints, and builds a timeline.
7. The script writes the JSON summary and Markdown report, then the tool reads the report back for the agent.

In LangSmith, inspect the tool call, OpenShell execution span, generated artifacts, and the evidence the agent used in the final response.

In [ ]:
run_agent(
    "Use the analyze_incident_log tool for the core incident-analysis lab. "
    "After the tool runs, summarize the generated /sandbox/incident_summary.json and "
    "/sandbox/incident_report.md artifacts, including the likely incident cause, evidence, "
    "and recommended next actions.",
    lab="python-incident-analysis",
)

## Lab 3: Inspect OpenShell Artifacts

Before moving on, inspect the files directly inside OpenShell. This reinforces the runtime boundary: the agent asked for an analysis, but the generated files live in the sandbox filesystem.

This cell uses the OpenShell CLI instead of the agent. You should see the same artifacts that Lab 2 referenced:

- `/sandbox/sample_app.log`
- `/sandbox/analyze_incident.py`
- `/sandbox/incident_summary.json`
- `/sandbox/incident_report.md`

This is useful in a real workflow because you can verify the sandbox state independently of the model's final answer.

In [ ]:
artifact_commands = [
    (
        "List sandbox files",
        ["uv", "run", "openshell", "sandbox", "exec", "--name", sandbox_name, "--no-tty", "--", "ls", "-lah", "/sandbox"],
    ),
    (
        "Pretty-print incident_summary.json",
        ["uv", "run", "openshell", "sandbox", "exec", "--name", sandbox_name, "--no-tty", "--", "python3", "-m", "json.tool", "/sandbox/incident_summary.json"],
    ),
    (
        "Preview incident_report.md",
        ["uv", "run", "openshell", "sandbox", "exec", "--name", sandbox_name, "--no-tty", "--", "sed", "-n", "1,80p", "/sandbox/incident_report.md"],
    ),
]

for label, command in artifact_commands:
    print(f"\n## {label}")
    run_command(command, check=True, timeout=120)

## Lab 4: Generic Sandbox Tooling

The incident-analysis lab used a purpose-built workshop tool. A general Deep Agent usually does not have one custom tool for every task. It uses the built-in file and execution tools that Deep Agents provides, while the backend decides where those operations happen.

In this exercise, ask the agent to create a small Python script in `/sandbox`, execute it, and summarize the result. Do not use `analyze_incident_log`; the goal is to watch the generic `write_file` -> `execute` -> result-summary pattern in LangSmith.

In LangSmith, compare this run with Lab 2. Lab 2 should show the custom `analyze_incident_log` path. This lab should show the reusable Deep Agents tooling path backed by the same OpenShell sandbox.


In [ ]:
run_agent(
    "Use the normal Deep Agents file and execution tools, not analyze_incident_log. "
    "Create /sandbox/generic_tooling_demo.py with Python code that prints compact JSON containing "
    "python_version, platform, cwd, count_1_to_25, and sum_1_to_25. "
    "Execute it in OpenShell, read back the output, and summarize the file path, command, and JSON result. "
    "Keep the final answer concise.",
    lab="generic-sandbox-tooling",
)


## Lab 5: Policy Observability

This run demonstrates why OpenShell policy controls are useful. One write to `/sandbox` should be allowed and one write to `/etc` should be denied. The denied operation is not a failure of the workshop; it is the point of the lab.

In [ ]:
run_agent(
    "Use the run_policy_boundary_demo tool for the workshop policy lab. "
    "Summarize which write was allowed, which write was denied, why that denial is expected, "
    "and which LangSmith spans to inspect.",
    lab="policy-observability",
)

### Policy Scope: Workshop Policy vs Dev-Agent Policy

This workbook intentionally uses a small policy so the allow/deny behavior is predictable. That is different from a broader coding-agent policy, where the product may intentionally allow package installs, GitHub reads, or selected API calls.

| Policy question | This workshop | Broader dev-agent policy |
| --- | --- | --- |
| Writable paths | `/sandbox`, `/tmp`, `/dev/null` | Often also `/workspace` or a project checkout |
| Network access | Not part of the core lab | Usually allowlisted by endpoint and binary, such as PyPI, GitHub, or NVIDIA inference |
| Goal | Teach observable sandbox boundaries | Enable real development while still constraining sensitive actions |
| Failure mode to inspect | `/etc` write denied | Package, repo, or API call denied by a narrower allowlist |

The important habit is the same in both cases: when an operation fails, inspect whether the cause was model behavior, Python code, command exit status, or OpenShell policy enforcement.


## Lab 6: Skills And Iteration

Deep Agents can load task-specific skills from `skills/`. Read the incident-analysis skill below, then edit `skills/incident-log-analysis/SKILL.md` in your editor.

Try adding an instruction like: `Always include a customer impact estimate and a confidence level in the report.`

After editing, run the next prompt and compare the guidance with Lab 2.

### Skills vs Memory

This workbook uses skills because they are explicit, reviewable, and reproducible for every participant. A production Deep Agent can also use persistent memory, such as a `/memory/AGENTS.md` route, to remember reliable patterns across runs.

That tradeoff matters:

- Use **skills** for curated behavior you want to teach, review, version, and share.
- Use **memory** when the agent should accumulate local operating knowledge over time.
- Keep memory optional in workshops, because persistent learned behavior can make two participants' runs diverge.


In [ ]:
skill_path = ROOT / "skills/incident-log-analysis/SKILL.md"
print(skill_path.read_text(encoding="utf-8")[:1600])

In [ ]:
run_agent(
    "Review the incident-log-analysis skill and the existing /sandbox/incident_report.md artifact. "
    "Explain how the skill should change to emphasize customer impact, confidence level, "
    "and specific remediation steps. Do not rerun analysis unless an artifact is missing.",
    lab="skill-iteration",
)

## Lab 7: Debugging With LangSmith

Use this checklist after you have at least one successful incident-analysis run and one policy-boundary run. Open your LangSmith project and inspect the traces from the previous cells.

In [ ]:
trace_debugging_checklist = [
    "Filter by tags: workshop, notebook, incident-analysis.",
    "Open the sandbox-smoke-test trace and confirm the OpenShell command execution span.",
    "Open the python-incident-analysis trace and inspect the analyze_incident_log tool call.",
    "Confirm /sandbox/incident_summary.json and /sandbox/incident_report.md were generated.",
    "Compare the direct OpenShell artifact-inspection output with the Lab 2 tool output.",
    "Open the policy-observability trace and compare the allowed /sandbox write with the denied /etc write.",
    "Use errors and exit codes to distinguish model behavior, sandbox execution, Python code, and policy enforcement.",
    "Compare thread ids and metadata fields: lab, sandbox_name, workshop, and use_case.",
]

for index, item in enumerate(trace_debugging_checklist, start=1):
    print(f"{index}. {item}")

## Wrap-Up

You have now walked through the intended workshop loop:

1. Configure local execution safely.
2. Run a Deep Agent from a notebook.
3. Analyze application logs by running Python inside OpenShell.
4. Produce concrete incident artifacts.
5. Use generic Deep Agents file and execution tools against the same sandbox.
6. Observe every meaningful step in LangSmith.
7. Use skills and traces to improve behavior.

The key takeaway: the agent is not a black box when you combine Deep Agents, OpenShell, and LangSmith. You can inspect what it planned, what it delegated, what code it executed, what policy allowed or denied, and how to improve the next run.

To turn this pattern into a real application:

- Add a `langgraph.json` entry if you want to run the same agent through LangGraph Studio or a dev server.
- Decide whether the app should use curated skills, persistent memory, or both.
- Choose a policy that fits the product: narrow for demos and sensitive workflows, broader for development agents that need package or repo access.
- Keep redacted OpenShell spans so execution, file movement, and policy denials remain visible in LangSmith without leaking secrets.
- Replace the sample log fixture with real data connectors or uploaded incident artifacts.
